## Yahtzee MAML Strategy Analysis

Post-training analysis of FOMAML-trained strategies across reward tasks.
Load evaluation trajectory parquet files from `data/` and analyse strategy patterns.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure src/ is on path for any direct imports (e.g. env.constants in analysis cells)
_src = Path().resolve().parent / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
traj_files = sorted(Path("data/trajectories").glob("*.parquet"))
ep_files = sorted(Path("data/episodes").glob("*.parquet"))

if not traj_files:
    raise FileNotFoundError("No parquet files in data/trajectories/. Run evaluate.py first.")

df_steps = pd.concat([pd.read_parquet(f) for f in traj_files], ignore_index=True)
df_episodes = pd.concat([pd.read_parquet(f) for f in ep_files], ignore_index=True)

print(f"Loaded {len(df_steps):,} step rows from {len(traj_files)} files")
print(f"Loaded {len(df_episodes):,} episode rows from {len(ep_files)} files")
print(f"\nStrategies: {sorted(df_episodes['strategy'].unique())}")
print(f"Meta steps: {sorted(df_episodes['meta_step'].unique())}")
df_episodes.head()

In [ ]:
strategies = sorted(df_episodes["strategy"].unique())
data_by_strategy = [df_episodes[df_episodes["strategy"] == s]["final_score"].values for s in strategies]

fig, ax = plt.subplots()
parts = ax.violinplot(data_by_strategy, showmedians=True)
ax.set_xticks(range(1, len(strategies) + 1))
ax.set_xticklabels(strategies, rotation=20, ha="right")
ax.set_ylabel("Final Score")
ax.set_title("Score Distribution per Strategy")
plt.tight_layout()
plt.show()

print(df_episodes.groupby("strategy")["final_score"].agg(["mean", "std", "min", "max"]).round(1))

In [ ]:
score_steps = df_steps[df_steps["phase"] == "score"].copy()
score_steps = score_steps[score_steps["round"] > 0]

fig, ax = plt.subplots()
for strategy in strategies:
    sub = score_steps[score_steps["strategy"] == strategy]
    mean_score = sub.groupby("round")["cumulative_score"].mean()
    ax.plot(mean_score.index, mean_score.values, label=strategy)

ax.set_xlabel("Round (scoring action number)")
ax.set_ylabel("Mean Cumulative Score")
ax.set_title("Score Progression by Round per Strategy")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
upper_rates = (
    df_episodes.groupby("strategy")["upper_bonus_total"]
    .apply(lambda x: (x > 0).mean())
    .reindex(strategies)
)

fig, ax = plt.subplots()
ax.bar(strategies, upper_rates.values)
ax.set_ylabel("Rate (fraction of episodes)")
ax.set_title("Upper Bonus Completion Rate per Strategy")
ax.set_ylim(0, 1)
for i, v in enumerate(upper_rates.values):
    ax.text(i, v + 0.01, f"{v:.1%}", ha="center", fontsize=9)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Mean cross-outs per episode per strategy
crossout_means = df_episodes.groupby("strategy")["n_cross_outs"].mean().reindex(strategies)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(strategies, crossout_means.values)
axes[0].set_ylabel("Mean cross-outs per episode")
axes[0].set_title("Cross-outs per Strategy")
axes[0].set_xticks(range(len(strategies)))
axes[0].set_xticklabels(strategies, rotation=20, ha="right")

# Which categories are crossed out most? (action_category where cross_out is True)
crossouts = df_steps[(df_steps["phase"] == "score") & (df_steps["cross_out"] == True)].copy()
if len(crossouts) > 0:
    from env.constants import CATEGORY_NAMES
    cat_counts = crossouts["action_category"].value_counts().sort_index()
    axes[1].bar(
        [CATEGORY_NAMES[i] for i in cat_counts.index],
        cat_counts.values,
    )
    axes[1].set_title("Which Categories Get Crossed Out")
    axes[1].set_ylabel("Count")
    plt.setp(axes[1].get_xticklabels(), rotation=40, ha="right")
else:
    axes[1].text(0.5, 0.5, "No cross-outs in data", ha="center", va="center", transform=axes[1].transAxes)
    axes[1].set_title("Which Categories Get Crossed Out")

plt.tight_layout()
plt.show()

In [ ]:
roll_steps = df_steps[df_steps["phase"] == "roll"]
entropy_by_step = (
    roll_steps.groupby(["meta_step", "strategy"])["action_entropy"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots()
for strategy in strategies:
    sub = entropy_by_step[entropy_by_step["strategy"] == strategy]
    ax.plot(sub["meta_step"], sub["action_entropy"], marker="o", label=strategy)

ax.set_xlabel("Meta Step (checkpoint)")
ax.set_ylabel("Mean Roll-Phase Entropy")
ax.set_title("Policy Entropy over Training (lower = more decisive)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sample up to 500 episodes per strategy for scatter readability
sampled = (
    df_episodes.groupby("strategy", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 500), random_state=42))
    .reset_index(drop=True)
)

# Merge final_score onto the last score step per episode to get value_estimate
last_score_step = (
    df_steps[df_steps["phase"] == "score"]
    .sort_values("turn")
    .groupby(["episode_id", "strategy", "meta_step"])
    .last()
    .reset_index()[["episode_id", "strategy", "meta_step", "value_estimate"]]
)
cal_df = sampled.merge(last_score_step, on=["episode_id", "strategy", "meta_step"], how="left")

fig, ax = plt.subplots()
for strategy in strategies:
    sub = cal_df[cal_df["strategy"] == strategy]
    ax.scatter(sub["value_estimate"], sub["final_score"], alpha=0.3, s=10, label=strategy)

lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1]),
]
ax.plot(lims, lims, "k--", linewidth=0.8, label="perfect calibration")
ax.set_xlabel("Value Estimate (critic)")
ax.set_ylabel("Actual Final Score")
ax.set_title("Value Estimate Calibration")
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()